In [6]:
%load_ext autoreload
%autoreload 2

import json
from dotenv import load_dotenv
from IPython.display import display, Markdown
import pandas as pd
from random import choice
import srsly
from textwrap import TextWrapper

def dump(data):
    print(json.dumps(data, indent=4, sort_keys=True))
 
def print_long(text, width=140, indent=0, initial_indent=0):
    wrapper = TextWrapper(initial_indent=" " * initial_indent, subsequent_indent=" " * indent, width=width)
    print("\n".join(wrapper.wrap(text)))

load_dotenv()

True

# Implement Call-04 Run Gate Utility

In [47]:
import requests

MODEL = "calcworks/finch-slm-fine-tuning-call-04-gate-smollm3-3b"

# Define the server URL and endpoints
URL = "http://localhost:8000/v1/chat/completions"

# Set up the headers
HEADERS = {
    "Content-Type": "application/json"
}

SYSTEM_MESSAGE = {
    "content": "You are the subject-check gate in a researcher URL-classification pipeline.\nGiven a candidate web page and a target researcher, decide whether the page is\nPRIMARILY ABOUT that specific researcher.\n\nDecide the verdict:\n- about_researcher : the page belongs to the target researcher's own web footprint.\n  This INCLUDES all of:\n    * their faculty/directory profile, personal site, blog, or CV/resume;\n    * a page representing a LABORATORY, RESEARCH GROUP, or CENTER that the researcher\n      leads, directs, is a principal investigator of, or is a named member of \u2014 even\n      when the page describes the entity rather than giving a biography of the person\n      (e.g. a lab homepage, a research-group site, a center the researcher belongs to\n      in their field);\n    * a news article or feature whose primary subject is the researcher.\n  A lab/group/center page counts here only when the target is genuinely part of it \u2014\n  NOT a generic university/department homepage they merely happen to be employed by.\n- different_person : the page is primarily about someone else (a different person with\n  the same or similar name, a co-author, a different faculty member).\n- passing_mention  : the target appears only incidentally on a page that is not part of\n  their footprint \u2014 a faculty/staff directory or roster listing many people, a co-author\n  or citation list, a student roster, or a page primarily about a DIFFERENT person that\n  happens to name the target.\n- uncertain        : the page content is insufficient to decide.\n\nJudge identity using the name AND the institution AND the field of study together.\nA page can name-match but be a DIFFERENT person if the institution or research field\nclearly conflicts (e.g. same name, but the page is a materials scientist while the\ntarget works on natural language processing \u2192 different_person).\n\nAlso report:\n- name_match : exact | partial | none  (does the researcher's name appear as the page subject)\n- field_match: match | mismatch | unknown | n/a  (does the page's field align with the target's topics)\n- reason     : one short sentence.\n",
    "role": "system"
}

def run_gate(content:str, max_tokens:int=512, verbose=False)->dict[str, str]:
        
    # Define the payload using OpenAI format
    data = {
        "model": MODEL,
        "messages": [
            SYSTEM_MESSAGE,
            {"role": "user", "content": content}
        ],
        "temperature": 0.0,
        "max_tokens": max_tokens
    }
    
    # Send the POST request
    response = requests.post(URL, headers=HEADERS, json=data)
    
    # Parse and print the response
    if response.status_code == 200:
        response_text = response.text 
        response = response.json()
        response_content = response["choices"][0]["message"]["content"]
        index = response_content.find("<think>") + len("<think>")
        json_content = response_content[index:].strip()
        try:
            json_content = json.loads(json_content)
            if verbose:
                print("RESPONSE:")
                dump(json_content)    
                print(f"CONTENT:\n{content}")
            json_content["error"] = False
            return json_content
        except:
            return {"error":True, "response":response_text}
    else:
            return {"error":True, "response":response.text}


# Run Test/Eval Samples

In [43]:
from random import choice

import srsly

path = "/workspace/ft/data/gate_test_eval.jsonl"

SAMPLES = srsly.read_jsonl(path)
SAMPLES = list(SAMPLES)
sample = choice(SAMPLES)
content = sample['messages'][-1]['content']
results = run_gate(content, max_tokens=512, verbose=True)

RESPONSE:
{
    "field_match": "mismatch",
    "name_match": "none",
    "reason": "The page is about Kristen Grauman, a different person from Eric Fosler-Lussier, and the field of computer vision/machine learning does not match Eric's speech processing/natural language processing focus.",
    "verdict": "different_person"
}
CONTENT:
TARGET RESEARCHER
  name: Eric Fosler-Lussier
  institution: Ohio State University
  department: Computer Science
  research topics: Speech Processing, Natural Language Processing, Machine Learning, Human-Computer Interaction

CANDIDATE PAGE
  url: https://ai.meta.com/people/1517773078782170/kristen-grauman/
  title: (none)
  content (may be truncated):
[![Meta](https://scontent.fzmm2-1.fna.fbcdn.net/v/t39.8562-6/252294889_575082167077436_6034106545912333281_n.svg/meta-logo-primary_standardsize.svg?_nc_cat=108&ccb=1-7&_nc_sid=e280be&_nc_ohc=AubU6T19KmIQ7kNvwFj64fz&_nc_oc=AdrHwxbLwOdm8K8lEPsCn42E2qkBfDw9URbnP9hGmlopUfmFDPU7x3pEMqb9YW3s5syaAjog_8JILjiVV8qFXq

In [45]:
results

{'verdict': 'different_person',
 'name_match': 'none',
 'field_match': 'mismatch',
 'reason': "The page is about Kristen Grauman, a different person from Eric Fosler-Lussier, and the field of computer vision/machine learning does not match Eric's speech processing/natural language processing focus.",
 'error': False}

# Run Simple Perf Test

In [67]:
import time
from tqdm import tqdm

start_time = time.time()
error_cnt = 0
cnt = 0
error_samples = []
with tqdm(SAMPLES, ncols=100) as pbar:
    for sample in pbar:
        content = sample['messages'][-1]
        results = run_gate(content, max_tokens=512, verbose=False)
        cnt += 1
        if results['error']: 
            error_cnt += 1
            error_samples.append(sample)
end_time = time.time()
execution_time = end_time - start_time
print(
    f"\nTask took {execution_time:.2f} seconds for {cnt:d} samples.\n"
    f"Average running time = {execution_time / cnt:.2f} seconds per sample.\n"
    f"There {'was one error' if error_cnt==1 else 'no errors' if error_cnt==0 else f'{error_cnt} errors'}"
) 


100%|██████████████████████████████████████████████████████████████| 62/62 [00:00<00:00, 209.56it/s]


Task took 0.30 seconds for 62 samples.
Average running time = 0.00 seconds per sample.
There 62 errors


In [76]:
sample = choice(error_samples)
#results = run_gate(sample['messages'][-1]['content'], max_tokens=512, verbose=False)
print(f"Has Error: {'Yes' if results['error'] else 'No'}")
print(json.loads(results["response"])['error']['message'].strip())

Has Error: Yes
4 validation errors for ValidatorIterator
0.ChatCompletionContentPartTextParam
  Input should be a valid dictionary [type=dict_type, input_value='role', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/dict_type
0.ChatCompletionContentPartImageParam
  Input should be a valid dictionary [type=dict_type, input_value='role', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/dict_type
0.ChatCompletionContentPartInputAudioParam
  Input should be a valid dictionary [type=dict_type, input_value='role', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/dict_type
0.File
  Input should be a valid dictionary [type=dict_type, input_value='role', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/dict_type
